## 1. Préparation de l'environnement et des données ESS

Je reprends la base d'avant en chargeant nos fichiers


In [37]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import random

In [38]:

# Chargement de la liste ESS (on ne garde que le SIREN pour la jointure)
df_ess_ref = pd.read_csv('C:\\Users\\Utilisateur\\OneDrive\\Documents\\centrale Lille + sc po\\Projet SOGA\\ess-numerique-database\\data\\raw\\entreprisesess.csv', usecols=['SIREN'], dtype={'siren': str})
ess_sirens = set(df_ess_ref['SIREN'])

print(f"Nombre de structures ESS référencées : {len(ess_sirens)}")

Nombre de structures ESS référencées : 1130252


In [46]:
file = pq.ParquetFile('C:\\Users\\Utilisateur\\OneDrive\\Documents\\centrale Lille + sc po\\Projet SOGA\\ess-numerique-database\\data\\raw\\StockUniteLegale_utf8.parquet')


## 2. Lecture optimisée du stock INSEE (Parquet)

Pour ne pas charger 30 Millions de lignes, nous allons utiliser une lecture sélective. Choix 2 :  Prendre les catégorie juridique [9220, 6316, 9300] qui sont associations, coopératives ou fondations, en plsu de travailler sur 10 chunks. Nous avons donc déjà un laissé pour compte : les ESUS

In [40]:
# Liste des colonnes nécessaires pour la cartographie
cols_to_keep = [
    'siren', 'denominationUniteLegale', 'nomUniteLegale', 
    'activitePrincipaleUniteLegale', 'trancheEffectifsUniteLegale', 'categorieJuridiqueUniteLegale'
]


In [41]:
ESS_CODES = [
    9220,  # Associations
    6316, 6317, 6318,  # Coopératives
    9300   # Fondations
]


In [42]:
codes_naf_numerique = [
    '62.01Z', '58.29A', '58.29B', '58.29C',
    '63.11Z'
]

In [43]:
ess_numerique_chunks = []

n_groups = min(10, file.num_row_groups)

for i in range(n_groups):
    print(f"Lecture row group {i+1}/{n_groups}")
    
    table = file.read_row_group(i, columns=cols_to_keep)
    df_chunk = table.to_pandas()
    
    df_filtered = df_chunk[
        (df_chunk['categorieJuridiqueUniteLegale'].isin(ESS_CODES)) &
        (df_chunk['activitePrincipaleUniteLegale'].isin(codes_naf_numerique))
    ]
    
    if not df_filtered.empty:
        ess_numerique_chunks.append(df_filtered)

df_ess_numerique = pd.concat(ess_numerique_chunks, ignore_index=True)

print(f"Nombre total (sur 10 row groups) : {len(df_ess_numerique)}")


Lecture row group 1/10
Lecture row group 2/10
Lecture row group 3/10
Lecture row group 4/10
Lecture row group 5/10
Lecture row group 6/10
Lecture row group 7/10
Lecture row group 8/10
Lecture row group 9/10
Lecture row group 10/10
Nombre total (sur 10 row groups) : 22


In [ ]:
df_ess_numerique

,siren,denominationUniteLegale,nomUniteLegale,activitePrincipaleUniteLegale,trancheEffectifsUniteLegale,categorieJuridiqueUniteLegale
0,301123741,CTRE STATISTIQ INFORMAT APPLIQUEES,None,63.11Z,NN,9220
1,301652673,INSTITUT TECH FEDERATION FRANC BATIMENT,None,63.11Z,12,9220
2,302044219,INSTITUT RECHERCHE L ENSAD,None,63.11Z,NN,9220
3,302997986,ASS DOCUMENTATION SCIENT INDUST COMMERCI,None,63.11Z,NN,9220
4,303891915,ASSOC FRANCAISE MICADO,None,63.11Z,02,9220
5,306016411,CENTRE CULTUREL LUTHERIEN DE PARIS,None,63.11Z,NN,9220
6,306169905,ASS CARREFOUR DU MONDE RURAL,None,63.11Z,NN,9220
7,306215179,FEDERATION ASSOCI REGIO PENELO,None,63.11Z,NN,9220
8,306387390,OFFICE POUR INFORMAT DOCUMENT APICULTURE,None,63.11Z,NN,9220
9,307167965,CELLULE ECONOM REGION AUVERGNE,None,63.11Z,NN,9220
